# Desafio 1 — Análise de Crédito (CredAIR)

**Contexto:** a CredAIR, uma financeira fictícia de empréstimos pessoais, mantém um histórico de propostas de crédito acompanhadas até a definição de inadimplência. O objetivo desta análise é compreender os fatores relacionados à inadimplência e investigar se as informações disponíveis permitem **identificar propostas com maior risco**.

**Estrutura do notebook** (espelhando as 8 etapas do enunciado):

1. Entendimento do problema
2. Análise exploratória dos dados
3. Limpeza e preparação dos dados
4. Engenharia de atributos
5. Preparação para modelagem
6. Modelagem (Regressão Logística)
7. Avaliação e interpretação
8. Conclusão

> Cada bloco de código é precedido de uma célula de texto explicando **o que** será feito e **por quê** a decisão foi tomada dessa forma.

## Configuração do ambiente e carregamento dos dados

**O que:** importamos as bibliotecas de manipulação e visualização de dados (`pandas`, `numpy`, `matplotlib`) e carregamos o arquivo `dataset.csv`.

**Por quê:** a base é pequena (5.000 linhas) e cabe confortavelmente em memória, então `pandas` puro é suficiente — não há necessidade de computação distribuída. Fixar o caminho do arquivo em uma variável (`CAMINHO_DADOS`) facilita a manutenção e a reprodutibilidade: qualquer pessoa com o repositório executa o notebook e obtém exatamente os mesmos dados.

In [0]:
# Importação das bibliotecas usadas em todo o notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Leitura do dataset (arquivo disponível no mesmo diretório do notebook)
CAMINHO_DADOS = "/Workspace/Repos/cardosoamelia2001@gmail.com/fast-track-ia-t1/desafios/analise-credito/dataset.csv"
df = pd.read_csv(CAMINHO_DADOS)

# Primeira visão: dimensões e amostra dos dados
print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()

---

## 1. Entendimento do problema

**O problema de negócio.** A CredAIR concedeu e acompanhou empréstimos pessoais ao longo do tempo e registrou, para cada proposta, o perfil do solicitante (idade, estado, escolaridade, renda, tempo de emprego), sua situação financeira (score de crédito, dívidas ativas) e as condições do empréstimo (valor, prazo, finalidade). O objetivo é usar esse histórico para **identificar propostas com maior risco de inadimplência**.

**Variável resposta: `inadimplente`.** É a variável que queremos prever. Ela indica se a proposta resultou (1) ou não (0) em inadimplência, após o acompanhamento do empréstimo. Por ser observada apenas *depois* da concessão, ela é o resultado do processo — exatamente o que o modelo deve antecipar no momento da proposta.

**Caracterização do problema de Machine Learning:**

* é **supervisionado**, pois dispomos de rótulos históricos (`inadimplente`) associados a cada proposta observada;
* é **classificação binária**, pois o alvo é uma categoria com dois valores possíveis (adimplente × inadimplente), e não uma quantidade contínua.

**Por que classificação (e não regressão)?** A pergunta de negócio é "esta proposta vai resultar em inadimplência?" — uma resposta do tipo sim/não. Estimar uma quantidade contínua (por exemplo, o prejuízo em reais) seria um problema de regressão, que não é o que se pede. Além disso, a classificação binária entrega uma **probabilidade de risco por proposta**, que a área de crédito pode usar para calibrar limites de aprovação, balanceando explicitamente aprovar mais (mais receita) contra inadimplir mais (mais perda).

In [0]:
# Distribuição da variável resposta: contagem e proporção de cada classe
contagem = df['inadimplente'].value_counts()
proporcao = (df['inadimplente'].value_counts(normalize=True) * 100).round(1)

print(f"Adimplentes (0):   {contagem[0]} propostas ({proporcao[0]}%)")
print(f"Inadimplentes (1): {contagem[1]} propostas ({proporcao[1]}%)")

---

## 2. Análise exploratória dos dados

**O que:** antes de qualquer modelagem, investigamos a estrutura dos dados: tipos de variáveis, distribuições, categorias existentes e as relações entre as variáveis explicativas e a variável resposta.

**Por quê:** a análise exploratória justifica as decisões das etapas seguintes — quais problemas de qualidade tratar (Seção 3), quais variáveis derivadas criam valor (Seção 4) e o que esperar do modelo (Seção 6). Modelar sem explorar é trabalhar no escuro.

### 2.1 Visão geral dos dados

**O que:** conferimos tipos, valores ausentes, estatísticas descritivas das numéricas e as categorias das qualitativas.

**Por quê:** tipos errados (ex.: números lidos como texto) quebrariam o modelo; valores ausentes exigiriam tratamento; e as estatísticas descritivas revelam faixas implausíveis — o primeiro passo para detectar os problemas de qualidade que serão tratados na Seção 3.

In [0]:
# Tipos de variáveis e presença de valores ausentes
df.info()

# Estatísticas descritivas das variáveis numéricas
df.describe().round(2)

In [0]:
# Categorias das variáveis qualitativas — atenção ao número de "estados"!
for col in ['estado', 'escolaridade', 'finalidade']:
    print(f"\n{col}: {df[col].nunique()} categorias distintas")
    print(df[col].value_counts().head(8))

### 2.2 Relação das variáveis numéricas com a inadimplência

**O que:** comparamos a média de cada variável numérica entre adimplentes e inadimplentes e calculamos a correlação de Pearson de cada variável com a resposta.

**Por quê:** diferenças relevantes entre os grupos são o primeiro sinal de poder preditivo; a correlação resume, em uma escala comparável, a direção (protetora ou de risco) e a intensidade de cada relação.

In [0]:
# Correlação de Pearson entre cada variável numérica e a inadimplência
num_cols = ['idade', 'renda_mensal', 'tempo_emprego_anos', 'score_credito',
            'valor_solicitado', 'prazo_meses', 'dividas_ativas']
print(df[num_cols + ['inadimplente']].corr()['inadimplente'].drop('inadimplente').sort_values().round(3))

# Médias das variáveis numéricas por classe da variável resposta
df.groupby('inadimplente')[num_cols].mean().round(2)

### 2.3 Relação das variáveis categóricas com a inadimplência

**O que:** calculamos a taxa de inadimplência por nível de escolaridade e por finalidade do empréstimo.

**Por quê:** para variáveis categóricas não existem média comparável nem correlação direta; a **taxa de inadimplência por grupo** é a forma natural de medir associação. Se um nível tiver taxa muito distante da média geral (~34%), ele carrega sinal preditivo; se todos os níveis ficarem próximos, a variável tende a ser pouco informativa.

In [0]:
# Taxa de inadimplência (%) por categoria
for col in ['escolaridade', 'finalidade']:
    taxa = (df.groupby(col)['inadimplente'].mean() * 100).round(1).sort_values(ascending=False)
    print(f"\n== Taxa de inadimplência por {col} (%)")
    print(taxa.to_string())

### 2.4 Visualizações

**O que:** histogramas comparando score de crédito e renda mensal entre as classes, boxplot do valor solicitado e a média de dívidas ativas por classe.

**Por quê:** as tabelas resumem, mas não mostram a forma das distribuições nem o quanto os grupos se sobrepõem. Visualizar deixa claro, por exemplo, que os inadimplentes concentram-se em scores menores, mas as distribuições ainda se sobrepõem bastante — o que antecipa que o modelo conseguirá capturar tendências, porém sem separar os grupos perfeitamente.

In [0]:
# Painel de visualizações comparando adimplentes e inadimplentes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma: score de crédito por classe
for classe, rotulo, cor in [(0, 'Adimplente', 'tab:blue'), (1, 'Inadimplente', 'tab:red')]:
    axes[0, 0].hist(df.loc[df['inadimplente'] == classe, 'score_credito'],
                    bins=30, alpha=0.6, label=rotulo, density=True, color=cor)
axes[0, 0].set_title('Score de crédito por classe')
axes[0, 0].set_xlabel('score_credito')
axes[0, 0].legend()

# Histograma: renda mensal por classe
for classe, rotulo, cor in [(0, 'Adimplente', 'tab:blue'), (1, 'Inadimplente', 'tab:red')]:
    axes[0, 1].hist(df.loc[df['inadimplente'] == classe, 'renda_mensal'],
                    bins=30, alpha=0.6, label=rotulo, density=True, color=cor)
axes[0, 1].set_title('Renda mensal por classe')
axes[0, 1].set_xlabel('renda_mensal (R$)')
axes[0, 1].legend()

# Boxplot: valor solicitado por classe
axes[1, 0].boxplot([df.loc[df['inadimplente'] == 0, 'valor_solicitado'],
                    df.loc[df['inadimplente'] == 1, 'valor_solicitado']])
axes[1, 0].set_xticklabels(['Adimplente', 'Inadimplente'])
axes[1, 0].set_title('Valor solicitado por classe')

# Barras: média de dívidas ativas por classe
medias_dividas = df.groupby('inadimplente')['dividas_ativas'].mean()
axes[1, 1].bar(['Adimplente', 'Inadimplente'], medias_dividas.values, color=['tab:blue', 'tab:red'])
axes[1, 1].set_title('Média de dívidas ativas por classe')

plt.tight_layout()
plt.show()

### Síntese da análise exploratória

Os dados têm **5.000 propostas e 11 variáveis, sem valores ausentes**, mas com problemas de qualidade que serão tratados na Seção 3: 19 linhas duplicadas, `estado` com 85 grafias distintas (espaços extras), idades fora de faixa plausível (máximo de 750 anos) e tempo de emprego maior do que a própria vida trabalhista do solicitante.

**Padrões encontrados em relação à inadimplência:**

* **Score de crédito** é a variável numérica com maior correlação (−0,30): inadimplentes têm, em média, score de **617 contra 671** dos adimplentes;
* **Renda mensal**: inadimplentes ganham em média **R$ 4.194 contra R$ 5.771** (correlação −0,19);
* **Dívidas ativas**: inadimplentes têm em média **2,3 dívidas contra 1,7** (correlação +0,17);
* **Valor e prazo**: inadimplentes pedem valores maiores (R$ 15.905 × R$ 13.405) com **prazos menores** (27,8 × 31,1 meses) — a combinação sugere **parcelas mensais mais pesadas**;
* Variáveis categóricas diferem pouco entre as classes: a taxa de inadimplência por finalidade varia apenas de 32% a 37%, e a escolaridade não apresenta gradiente claro;
* A idade tem correlação praticamente nula (−0,02).

Esses sinais são coerentes com a intuição de crédito: **menor score, menor renda, mais dívidas e parcelas mais pesadas → maior risco**. A pista "valor maior com prazo menor" motiva diretamente a variável derivada da Seção 4.

---

## 3. Limpeza e preparação dos dados

**Problemas de qualidade identificados e confirmados na exploração:**

| # | Problema | Tamanho | Decisão |
|---|---------|---------|--------|
| 1 | `estado` com 85 categorias por espaços extras (`' DF'`, `'DF '`) | 85 → 27 estados reais | Padronizar com `strip().upper()` |
| 2 | Linhas 100% duplicadas | 19 (0,4% da base) | Remover |
| 3 | `idade` fora da faixa plausível (18–100), máximo de 750 | 15 registros | Remover |
| 4 | `tempo_emprego_anos` > idade − 16 (impossível trabalhar antes dos ~16 anos) | 848 na base bruta (17%); 844 após os passos anteriores | Limitar ao teto idade − 16 |
| 5 | Valores ausentes | 0 | Nenhum tratamento necessário |

**Justificativa de cada decisão:**

* **(1) Padronização de `estado`:** os 85 valores são, na prática, 27 estados escritos com espaços à esquerda/direita. Sem o `strip()`, o one-hot encoding criaria colunas artificiais (`'DF'` ≠ `' DF'`) e o modelo perderia sinal;
* **(2) Duplicatas:** registros idênticos em todas as 11 colunas são quase certamente o mesmo evento gravado duas vezes (erro de carga). Mantê-los inflaria artificialmente a confiança das estimativas e, pior, poderia fazer a mesma proposta cair no treino **e** no teste — vazamento de dados (data leakage);
* **(3) Idades inválidas:** são 0,3% da base, com valores impossíveis (ex.: 750 anos). Não há como recuperar o valor verdadeiro; preencher ou "cortar" (clip) inventaria informação. A remoção é a opção mais honesta, com impacto mínimo;
* **(4) Tempo de emprego:** 848 registros (17% da base bruta) têm valor logicamente impossível; 4 deles estavam entre as duplicatas e idades inválidas removidas nos passos anteriores, restando 844 ajustados. Remover esses registros causaria perda grande de informação; em vez disso, aplicamos o **teto plausível** (idade − 16), tratando o excedente como erro de preenchimento. Limitação documentada: para esses registros, o valor passa a ser uma aproximação conservadora.

**Tipos e formatos:** os tipos já estão adequados (numéricas como `int`/`float`, categóricas como texto) e não há valores ausentes — verificado na EDA e reconfirmado ao final da limpeza abaixo.

In [0]:
print(f"Antes da limpeza: {df.shape[0]} linhas")

# 1) Padroniza a sigla do estado: remove espaços extras e usa caixa alta
n_antes = df['estado'].nunique()
df['estado'] = df['estado'].str.strip().str.upper()
print(f"Estados distintos: {n_antes} -> {df['estado'].nunique()}")

# 2) Remove linhas totalmente duplicadas
n_duplicadas = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicatas removidas: {n_duplicadas}")

# 3) Remove idades fora da faixa plausível (18 a 100 anos)
n_idade = ((df['idade'] < 18) | (df['idade'] > 100)).sum()
df = df[(df['idade'] >= 18) & (df['idade'] <= 100)]
print(f"Registros com idade inválida removidos: {n_idade}")

# 4) Limita o tempo de emprego ao máximo plausível (idade - 16)
n_emprego = (df['tempo_emprego_anos'] > df['idade'] - 16).sum()
df['tempo_emprego_anos'] = df['tempo_emprego_anos'].clip(upper=df['idade'] - 16)
print(f"Registros com tempo de emprego ajustado: {n_emprego}")

# Confere o resultado final da limpeza
print(f"\nDepois da limpeza: {df.shape[0]} linhas | Valores ausentes: {df.isna().sum().sum()}")

---

## 4. Engenharia de atributos

**O que:** criamos duas variáveis derivadas:

1. **`parcela_mensal` = valor_solicitado / prazo_meses** — valor aproximado da parcela mensal do empréstimo;
2. **`comprometimento_renda` = parcela_mensal / renda_mensal** — fração da renda mensal comprometida com a parcela.

**Como foram construídas:** divisões diretas entre variáveis já existentes. Não usamos juros porque a taxa contratual não está disponível nos dados — a parcela é uma aproximação linear, o que é suficiente para capturar a ideia de esforço de pagamento.

**O que representam:** `parcela_mensal` mede o **tamanho absoluto** da obrigação mensal; `comprometimento_renda` mede o **tamanho relativo ao orçamento** do solicitante — um empréstimo de R$ 500/mês é muito mais arriscado para quem ganha R$ 1.500 do que para quem ganha R$ 15.000.

**Relação esperada com a inadimplência:** quanto maior o comprometimento da renda, menor a folga financeira para absorver imprevistos — espera-se, portanto, **maior risco de inadimplência**. A EDA reforçou essa hipótese: inadimplentes pedem valores maiores com prazos menores (parcelas mais pesadas).

**Serão usadas no modelo?** Sim — a verificação empírica abaixo (taxa de inadimplência por quartil de comprometimento) mostra relação **fortemente monotônica** com a resposta, o que justifica incluí-las na modelagem.

In [0]:
# Variável 1: parcela mensal aproximada (sem juros)
df['parcela_mensal'] = df['valor_solicitado'] / df['prazo_meses']

# Variável 2: comprometimento da renda com a parcela
df['comprometimento_renda'] = df['parcela_mensal'] / df['renda_mensal']

# Verificação empírica: taxa de inadimplência por quartil de comprometimento
faixas = pd.qcut(df['comprometimento_renda'], q=4)
verificacao = df.groupby(faixas, observed=True)['inadimplente'].agg(taxa_inadimplencia='mean', n_propostas='count')
verificacao['taxa_inadimplencia'] = (verificacao['taxa_inadimplencia'] * 100).round(1)
verificacao

---

## 5. Preparação para modelagem

**Decisões e justificativas:**

* **Variáveis utilizadas:** todas as 10 variáveis explicativas originais (após a limpeza) mais as 2 derivadas. Não descartamos variável a priori — a Regressão Logística tem regularização suave (L2, padrão do sklearn) que lida bem com preditores fracos, e manter todas permite **interpretar** quais, de fato, importam;
* **Categóricas (`estado`, `escolaridade`, `finalidade`):** transformadas em variáveis indicadoras (one-hot). A regressão logística só trabalha com números, e o one-hot preserva a informação sem inventar ordem entre categorias (ao contrário de codificações ordinais arbitrárias);
* **Divisão treino/teste 80/20 com `stratify=y`:** as classes são desbalanceadas (65,6% × 34,4%), então uma divisão puramente aleatória poderia gerar um conjunto de teste com proporção de classes distorcida. A estratificação garante que treino e teste reflitam a mesma mistura — essencial para uma avaliação justa;
* **`random_state=42`:** fixa a divisão, garantindo reprodutibilidade (mesma divisão a cada execução);
* **Pré-processamento dentro do `Pipeline` (Seção 6):** padronização e one-hot serão ajustados **apenas no conjunto de treino**. Se padronizássemos usando estatísticas da base inteira, informação do teste vazaría para o treinamento (data leakage) e a avaliação seria otimista demais.

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            roc_auc_score, confusion_matrix, classification_report)

# X = variáveis explicativas, y = variável resposta
X = df.drop(columns=['inadimplente'])
y = df['inadimplente']

# Divisão estratificada 80/20
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Treino: {X_treino.shape[0]} propostas | Inadimplência: {y_treino.mean()*100:.1f}%")
print(f"Teste:  {X_teste.shape[0]} propostas | Inadimplência: {y_teste.mean()*100:.1f}%")

---

## 6. Modelagem — Regressão Logística

**O que:** treinamos uma **Regressão Logística** dentro de um `Pipeline` com dois pré-processadores:

* **Numéricas → `StandardScaler`** (padronização: média 0, desvio-padrão 1). **Por quê:** a regressão logística combina as variáveis por soma ponderada; com renda em milhares e dívidas em unidades, os coeficientes ficam em escalas incomparáveis e a otimização converge mal. Padronizar também torna os coeficientes **diretamente comparáveis em magnitude** — justamente o que a interpretação da Seção 7 exige;
* **Categóricas → `OneHotEncoder`** com `handle_unknown='ignore'`, para que uma categoria rara presente apenas no teste não faça o pipeline falhar.

**Por que Regressão Logística?** Além de ser o modelo pedido no desafio, é o clássico de análise de crédito: entrega **probabilidade de inadimplência** e coeficientes interpretáveis (odds ratios) — a CredAIR precisa *entender* os fatores de risco, não apenas prever.

**Processo de treinamento:** o `Pipeline` é ajustado (`fit`) apenas no conjunto de treino; o scaler aprende média/desvio do treino, o encoder aprende as categorias do treino, e a regressão ajusta seus coeficientes por máxima verossimilhança com regularização L2.

In [0]:
# Colunas por tipo
colunas_categoricas = ['estado', 'escolaridade', 'finalidade']
colunas_numericas = [c for c in X.columns if c not in colunas_categoricas]

# Pré-processamento: padroniza numéricas e codifica categóricas
preprocessamento = ColumnTransformer(transformers=[
    ('num', StandardScaler(), colunas_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), colunas_categoricas),
])

# Pipeline completo: pré-processamento + modelo
modelo = Pipeline(steps=[
    ('preprocessamento', preprocessamento),
    ('regressao_logistica', LogisticRegression(max_iter=1000)),
])

# Treinamento (apenas com os dados de treino)
modelo.fit(X_treino, y_treino)
print(f"Modelo treinado com {X_treino.shape[0]} propostas e "
      f"{len(modelo.named_steps['preprocessamento'].get_feature_names_out())} variáveis após o one-hot.")

---

## 7. Avaliação e interpretação

**Métricas utilizadas e por quê:**

* **Acurácia:** proporção de acertos no geral. Sozinha engana em base desbalanceada — prever sempre "adimplente" já acerta ~65,6%;
* **Precision** (entre os previstos como inadimplentes, quantos realmente são): errar aqui = **recusar bons clientes** (perda de receita);
* **Recall** (entre os verdadeiros inadimplentes, quantos o modelo captura): errar aqui = **aprovar maus pagadores** (prejuízo direto);
* **F1-score:** média harmônica de precision e recall, equilibra os dois tipos de erro;
* **ROC-AUC:** capacidade de **ranquear** propostas por risco (1,0 = perfeito; 0,5 = aleatório). É a métrica mais alinhada ao uso: a CredAIR aplicará um corte na probabilidade, e o AUC mede a qualidade do ranqueamento independentemente do corte.

**Referência simples (baseline):** um modelo trivial que prevê sempre a classe majoritária ("adimplente"). Qualquer modelo útil precisa superá-lo — caso contrário, a complexidade não se paga.

In [0]:
# Previsões no conjunto de teste (dados que o modelo nunca viu)
pred = modelo.predict(X_teste)
prob = modelo.predict_proba(X_teste)[:, 1]  # probabilidade de inadimplência

# Métricas da Regressão Logística
metricas = {
    'Acurácia': accuracy_score(y_teste, pred),
    'Precision': precision_score(y_teste, pred),
    'Recall': recall_score(y_teste, pred),
    'F1-score': f1_score(y_teste, pred),
    'ROC-AUC': roc_auc_score(y_teste, prob),
}
for nome, valor in metricas.items():
    print(f"{nome:10s}: {valor:.3f}")

# Matriz de confusão (linhas = classe real, colunas = classe prevista)
print("\nMatriz de confusão:")
print(confusion_matrix(y_teste, pred))

# Baseline: prevê sempre a classe majoritária
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_treino, y_treino)
print(f"\nAcurácia do baseline (classe majoritária): {accuracy_score(y_teste, baseline.predict(X_teste)):.3f}")

In [0]:
# Interpretação: coeficientes como razão de chances (odds ratio)
# OR > 1 aumenta a chance de inadimplência; OR < 1 reduz.
# As numéricas estão padronizadas, então o OR delas vale por +1 desvio-padrão na variável.
nomes = modelo.named_steps['preprocessamento'].get_feature_names_out()
odds_ratios = pd.Series(
    np.exp(modelo.named_steps['regressao_logistica'].coef_[0]),
    index=[n.split('__', 1)[1] for n in nomes],
).sort_values(ascending=False)

print("== Maiores fatores de RISCO (OR > 1) ==")
print(odds_ratios.head(8).round(3).to_string())
print("\n== Maiores fatores de PROTEÇÃO (OR < 1) ==")
print(odds_ratios.tail(8).sort_values().round(3).to_string())

### Interpretação dos resultados

**Desempenho.** A Regressão Logística atingiu **acurácia de 0,74**, superando o baseline de 0,66 — ou seja, aprendeu padrão real, e não apenas a frequência das classes. O **ROC-AUC de 0,78** indica bom poder de ranqueamento: em cerca de 78% das comparações entre um adimplente e um inadimplente, o modelo atribui maior probabilidade ao inadimplente.

**Trade-off precision × recall.** No corte padrão (probabilidade ≥ 50%): precision ~0,69 e recall ~0,45 — o modelo é conservador: quando aponta inadimplência, costuma acertar (69%), mas deixa passar cerca de 55% dos inadimplentes. Para a CredAIR, isso significa **recusar poucos bons clientes** ao custo de aprovar maus pagadores. Se o custo de uma inadimplência for alto, vale **abaixar o limiar** para capturar mais inadimplentes (mais recall) aceitando mais falsos alarmes — decisão de negócio, por isso entregamos a probabilidade, não apenas o rótulo.

**Variáveis mais relacionadas à inadimplência (odds ratios):**

* **`comprometimento_renda` (OR ≈ 2,5)** — o mais forte fator de risco: cada aumento de 1 desvio-padrão no comprometimento da renda **mais que dobra** a chance de inadimplência. Confirma o padrão da Seção 4 (taxa de 22% no quartil mais baixo → 55% no mais alto);
* **`dividas_ativas` (OR ≈ 1,6)** — mais dívidas em aberto, mais risco;
* **`score_credito` (OR ≈ 0,44)** — o mais forte fator de proteção: quanto maior o score, menor a chance;
* **`renda_mensal` (OR ≈ 0,74)** e **`tempo_emprego_anos` (OR ≈ 0,80)** — renda maior e estabilidade no emprego reduzem o risco;
* Estados e finalidades aparecem com efeitos menores e espalhados (ex.: RS, OR ≈ 1,4). Como a EDA não mostrou padrão claro nessas variáveis, parte desses efeitos deve ser ruído — com 37 colunas dummies, algum efeito aparece por acaso.

**Para a CredAIR:** os fatores de risco identificados são coerentes com a prática de crédito — comprometimento elevado da renda, dívidas em aberto, score baixo e renda baixa. O modelo formaliza esse conhecimento em uma **probabilidade por proposta**, que pode apoiar — mas não substituir — a decisão de crédito.

---

## 8. Conclusão

### O que foi feito

Partimos do dado bruto (5.000 propostas) e chegamos a um modelo de Regressão Logística que estima a **probabilidade de inadimplência** de novas propostas, percorrendo exploração, limpeza, engenharia de atributos e avaliação justa em dados nunca vistos no treinamento.

### Principais padrões encontrados

* A inadimplência atinge **34,4%** das propostas — carteira significativamente arriscada;
* Inadimplentes têm, em média, **score menor** (617 × 671), **renda menor** (R$ 4,2 mil × R$ 5,8 mil), **mais dívidas ativas** (2,3 × 1,7) e pedem **valores maiores com prazos menores** (parcelas mais pesadas);
* O atributo derivado **comprometimento de renda** mostrou a relação mais forte de todas: a taxa de inadimplência sobe de **22%** (quartil mais baixo) para **55%** (quartil mais alto);
* Escolaridade, finalidade e estado carregam pouco sinal preditivo.

### Fatores de risco mais relevantes (odds ratios)

1. **Comprometimento da renda com a parcela** — principal fator de risco (OR ≈ 2,5);
2. **Dívidas ativas** (OR ≈ 1,6);
3. **Score de crédito** — principal fator de proteção (OR ≈ 0,44);
4. **Renda mensal** (OR ≈ 0,74) e **tempo de emprego** (OR ≈ 0,80).

### Resultados do modelo (conjunto de teste)

| Métrica | Valor | Baseline |
|---------|-------|----------|
| Acurácia | 0,74 | 0,66 |
| ROC-AUC | 0,78 | 0,5 (aleatório) |
| Precision | 0,69 | — |
| Recall | 0,45 | — |
| F1-score | 0,55 | — |

O modelo ranqueia bem as propostas por risco, mas é conservador no corte padrão de 50%. **Ajustar o limiar** (por exemplo, priorizando recall) é uma decisão de negócio que depende do custo de recusar um bom cliente versus aprovar um inadimplente.

### Limitações da análise

* **Dados sintéticos** e de um único corte: não há informação de custo da inadimplência, taxa de juros, histórico de pagamentos nem evolução temporal;
* **Correlação, não causalidade:** o modelo captura associações — reduzir o prazo de um empréstimo não tornaria o cliente menos inadimplente, por exemplo;
* **Qualidade dos dados:** 17% da base tinha tempo de emprego logicamente impossível (limitamos ao teto idade − 16) e 0,3% foi descartado por idade inválida — parte da base é, portanto, aproximada;
* A Regressão Logística captura relações (aproximadamente) lineares — interações e não-linearidades ficam de fora.

### Cuidados para uso em situação real

* O modelo **não substitui** a política de crédito nem exigências regulatórias — é uma ferramenta de apoio à decisão;
* **Viés e justiça:** variáveis como estado e escolaridade podem carregar correlações indesejadas com perfis socioeconômicos; o impacto sobre grupos deve ser auditado antes de qualquer uso;
* **Monitoramento contínuo:** o desempenho decai com a mudança do comportamento dos clientes (deriva de dados) — exige reavaliação e retreino periódicos;
* **Calibragem do limiar:** o corte de 50% não é necessariamente o melhor; deve ser calibrado com o custo financeiro de cada tipo de erro.

---

*Notebook desenvolvido para o Desafio 1 — Análise de Crédito (CredAIR). Todas as decisões estão justificadas nas células de texto ao longo das seções.*